# Week 48, Day 4: Palindromes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w048d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
s = "forgeeksskeegfor"

def grow(lo, hi):
    while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
        lo, hi = lo - 1, hi + 1
    return s[lo + 1:hi]

print(grow(7, 7))      # centred on one character
print(grow(7, 8))      # centred between two characters

In [ ]:
def longest_palindrome(s):
    best = ""
    for centre in range(2 * len(s) - 1):
        lo = centre // 2
        hi = lo + centre % 2               # odd centres: lo == hi; even: hi = lo + 1
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        if hi - lo - 1 > len(best):
            best = s[lo + 1:hi]
    return best

def count_palindromes(s):
    total = 0
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, centre // 2 + centre % 2
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            total += 1                     # every successful step is one more palindrome
            lo, hi = lo - 1, hi + 1
    return total

print(longest_palindrome("forgeeksskeegfor"), longest_palindrome("babad"), repr(longest_palindrome("")))
print(count_palindromes("aaa"), count_palindromes("abc"))

In [ ]:
import random

M = (1 << 61) - 1
B = random.randrange(256, 1 << 40)

def prefix(s):
    h, pw = [0] * (len(s) + 1), [1] * (len(s) + 1)
    for i, ch in enumerate(s):
        h[i + 1] = (h[i] * B + ord(ch)) % M
        pw[i + 1] = pw[i] * B % M
    return h, pw

s = "abacabadabacaba"
n = len(s)
hf, pw = prefix(s)
hr, _ = prefix(s[::-1])

def get(h, i, j):
    return (h[j] - h[i] * pw[j - i]) % M

def is_pal(i, j):
    return get(hf, i, j) == get(hr, n - j, n - i)

print(is_pal(0, 7), is_pal(0, 15), is_pal(1, 6), is_pal(4, 11))

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

def shortest_palindrome(s):
    k = prefix_function(s + "#" + s[::-1])[-1]       # longest palindromic prefix
    return s[k:][::-1] + s

for s in ["abacd", "aacecaaa", "abcd", "racecar", ""]:
    print(repr(s), "->", repr(shortest_palindrome(s)))

In [ ]:
from collections import Counter

def can_rearrange(s):
    odd = sum(1 for c in Counter(s).values() if c % 2)
    return odd <= 1

def arrange(s):
    counts = Counter(s)
    half = "".join(ch * (c // 2) for ch, c in sorted(counts.items()))
    middle = "".join(ch for ch, c in counts.items() if c % 2)
    return half + middle + half[::-1] if len(middle) <= 1 else None

print(can_rearrange("carerac"), arrange("carerac"), can_rearrange("abc"), arrange("aabbc"))

> **Key idea.** Every palindrome has one of 2n − 1 centres; growing from each finds the longest palindromic substring and counts them all in O(n²) time and O(1) space. Forward and backward prefix hashes test any substring in O(1), so binary search on the radius gives O(n log n). The shortest palindrome made by adding to the front keeps the longest palindromic prefix, found from the prefix function of `s + "#" + reverse(s)`. Letters can be rearranged into a palindrome when at most one count is odd. Palindrome pairs come from splitting each word and looking up reversed pieces in a dict. Manacher's algorithm reuses mirrored radii inside the rightmost palindrome to run in O(n).

### Exercise: Expanding around centres

Write `longest_palindrome(s)` (the first one, leftmost, if several have the greatest length; `""` for the empty string) and `count_palindromes(s)` (every palindromic substring, counted by position, so "aaa" has 6), both by expanding around all 2n − 1 centres. Then `all_maximal(s)`: for each centre index c from 0 to 2n − 2, the length of the longest palindrome centred there (0 for an even centre between two different characters).

In [ ]:
def longest_palindrome(s):
    ___

def count_palindromes(s):
    ___

def all_maximal(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef is_pal(t):\n    return t == t[::-1]\ndef test_centres():\n    \"centres\"\n    assert longest_palindrome(\"forgeeksskeegfor\") == \"geeksskeeg\" and longest_palindrome(\"babad\") == \"bab\" and longest_palindrome(\"\") == \"\"\n    assert count_palindromes(\"aaa\") == 6 and count_palindromes(\"abc\") == 3 and count_palindromes(\"\") == 0\n    assert all_maximal(\"abba\") == [1, 0, 1, 4, 1, 0, 1] and all_maximal(\"\") == []\n    rng = random.Random(1)\n    for _ in range(200):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 15)))\n        subs = [(i, j) for i in range(len(s)) for j in range(i + 1, len(s) + 1) if is_pal(s[i:j])]\n        assert count_palindromes(s) == len(subs)\n        best = max(subs, key=lambda p: (p[1] - p[0], -p[0]), default=(0, 0))\n        assert longest_palindrome(s) == s[best[0]:best[1]]")

### Exercise: O(1) palindrome tests

Write `PalindromeChecker(s)` using forward and backward prefix hashes (random base, modulus 2⁶¹ − 1), with `is_pal(i, j)` for `s[i:j]` in O(1). Then `longest_pal_prefix_lengths(s, queries)`: for each `(i, j)` in `queries`, the length of the longest palindromic **prefix** of `s[i:j]`, checking lengths from longest down with `is_pal` (fine for short ranges).

In [ ]:
import random

class PalindromeChecker:
    def __init__(self, s):
        ___

def longest_pal_prefix_lengths(s, queries):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_checker():\n    \"checks\"\n    s = \"abacabadabacaba\"\n    pc = PalindromeChecker(s)\n    assert pc.is_pal(0, 7) and pc.is_pal(0, 15) and not pc.is_pal(1, 5) and pc.is_pal(3, 3) and pc.is_pal(5, 6)\n    rng = random.Random(2)\n    t = \"\".join(rng.choice(\"ab\") for _ in range(300))\n    pc = PalindromeChecker(t)\n    for _ in range(3000):\n        i = rng.randrange(301)\n        j = rng.randrange(i, 301)\n        assert pc.is_pal(i, j) == (t[i:j] == t[i:j][::-1])\n    assert longest_pal_prefix_lengths(\"abacabx\", [(0, 7), (1, 7), (6, 7), (2, 2)]) == [3, 5, 1, 0]")

### Exercise: The shortest palindrome

Write `shortest_palindrome(s)` (adding characters to the front only) in O(n) with a prefix function, and `shortest_palindrome_end(s)`: the shortest palindrome made by adding characters to the **end** only.

In [ ]:
def prefix_function(s):
    ___

def shortest_palindrome(s):
    ___

def shortest_palindrome_end(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow_front(s):\n    for k in range(len(s), -1, -1):\n        if s[:k] == s[:k][::-1]:\n            return s[k:][::-1] + s\ndef test_shortest():\n    \"palindromes\"\n    assert shortest_palindrome(\"abacd\") == \"dcabacd\" and shortest_palindrome(\"aacecaaa\") == \"aaacecaaa\" and shortest_palindrome(\"\") == \"\"\n    assert shortest_palindrome_end(\"abacd\") == \"abacdcaba\" and shortest_palindrome_end(\"dcaba\") == \"dcabacd\"\n    rng = random.Random(3)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 12)))\n        assert shortest_palindrome(s) == slow_front(s)\n        e = shortest_palindrome_end(s)\n        assert e == e[::-1] and e.startswith(s) and len(e) == len(slow_front(s[::-1]))\n    s = \"a\" * 100_000 + \"b\" + \"a\" * 99_999\n    t0 = time.perf_counter()\n    r = shortest_palindrome(s)\n    assert len(r) == 300_000 and r == r[::-1] and time.perf_counter() - t0 < 2")

### Exercise: Rearranging into a palindrome

Write `can_rearrange(s)` and `smallest_arrangement(s)`: the alphabetically smallest palindrome using exactly the letters of s, or `None` if there's none. Then `fewest_changes(s)`: the fewest single-character replacements that turn s into a palindrome (without rearranging), using two pointers.

In [ ]:
from collections import Counter

def can_rearrange(s):
    ___

def smallest_arrangement(s):
    ___

def fewest_changes(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_rearrange():\n    \"rearranging\"\n    assert can_rearrange(\"carerac\") and not can_rearrange(\"abc\") and can_rearrange(\"\")\n    assert smallest_arrangement(\"carerac\") == \"acrerca\" and smallest_arrangement(\"abc\") is None and smallest_arrangement(\"\") == \"\"\n    rng = random.Random(4)\n    for _ in range(100):\n        s = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 6)))\n        pals = sorted({\"\".join(p) for p in itertools.permutations(s) if \"\".join(p) == \"\".join(p)[::-1]})\n        assert can_rearrange(s) == bool(pals) and smallest_arrangement(s) == (pals[0] if pals else None)\n    assert fewest_changes(\"abcd\") == 2 and fewest_changes(\"abca\") == 1 and fewest_changes(\"racecar\") == 0")

### Exercise: Palindrome pairs

Given a list of distinct words, write `palindrome_pairs(words)`: every ordered pair `(i, j)` with `i != j` such that `words[i] + words[j]` is a palindrome, sorted. Do it in O(n k²) with a dict from word to index: for each word and each split point, check the two cases described in "Rearranging and pairing" (take care not to report a pair twice when a split is at either end).

In [ ]:
def palindrome_pairs(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(words):\n    return sorted((i, j) for i in range(len(words)) for j in range(len(words)) if i != j and (words[i] + words[j]) == (words[i] + words[j])[::-1])\ndef test_pairs():\n    \"pairs\"\n    assert palindrome_pairs([\"abcd\", \"dcba\", \"lls\", \"s\", \"sssll\"]) == [(0, 1), (1, 0), (2, 4), (3, 2)]\n    assert palindrome_pairs([\"a\", \"\"]) == [(0, 1), (1, 0)] and palindrome_pairs([]) == []\n    rng = random.Random(5)\n    for _ in range(200):\n        words = list({\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 4))) for _ in range(rng.randint(0, 8))})\n        assert palindrome_pairs(words) == slow(words), words")

### Bug hunt: Only odd centres

This tries only the n centres on characters, so it never finds an even-length palindrome: this prints `a`. Try the centres between characters too, so it prints `abba`.

In [ ]:
def longest_palindrome(s):
    best = ""
    for c in range(len(s)):
        lo, hi = c, c
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        if hi - lo - 1 > len(best):
            best = s[lo + 1:hi]
    return best

print(longest_palindrome("xabbay"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"even centres\"\n    expect_output(\"abba\")")

### Bug hunt: A border across the join

This finds the longest palindromic prefix from the prefix function of `s + reverse(s)` with no separator. For "aaba", the border of "aabaabaa" runs across the join and comes out longer than the string itself, so nothing gets added and this prints `aaba`, which isn't a palindrome. Put `"#"` between the two halves so it prints `abaaba`.

In [ ]:
def prefix_function(s):
    pi, k = [0] * len(s), 0
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = pi[k - 1]
        if s[i] == s[k]:
            k += 1
        pi[i] = k
    return pi

def shortest_palindrome(s):
    k = prefix_function(s + s[::-1])[-1]
    return s[k:][::-1] + s

print(shortest_palindrome("aaba"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a separator\"\n    expect_output(\"abaaba\")")

### Bug hunt: Counting the wrong thing

This should count palindromic substrings, but it adds 1 per **centre** that has any palindrome at all, instead of 1 per successful step: this prints `5` for "aaa". Count every step so it prints `6`.

In [ ]:
def count_palindromes(s):
    total = 0
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, centre // 2 + centre % 2
        found = False
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            found = True
            lo, hi = lo - 1, hi + 1
        if found:
            total += 1
    return total

print(count_palindromes("aaa"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every step\"\n    expect_output(\"6\")")

### Puzzle: Growing from a centre

Arrange the lines into the longest-palindrome search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # how many centres a string of length 6 has
q = ___   # the longest palindromic substring of "cbbd"
r = ___   # the number of palindromic substrings of "abba"
s = ___   # the shortest palindrome made by adding to the front of "aba" + "c"
t = ___   # True or False: can "aabbcd" be rearranged into a palindrome?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 11 and q == \"bb\" and r == 6, \"p, q and r\"\n    assert s == \"cabac\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: Manacher's algorithm

Write `manacher(s)` returning the list `d` for the transformed string `t = "#" + "".join(ch + "#" for ch in s)` ("abba" becomes "#a#b#b#a#", always of length 2n + 1): `d[i]` is the number of characters on **each side** of position i in the longest palindrome of t centred there (so `d[i]` is also the length of the corresponding palindrome in s). Use the mirror trick described in "Manacher's algorithm" to make it O(n).

Then use it for `longest_palindrome(s)` (leftmost on a tie), `count_palindromes(s)` (each centre i contributes `(d[i] + 1) // 2` palindromes of s), and `palindromic_prefix_ends(s)`: a sorted list of every length L ≥ 1 for which `s[:L]` is a palindrome. The checks compare with expansion on random strings and time a 300,000-character string.

In [ ]:
def manacher(s):
    ___

def longest_palindrome(s):
    ___

def count_palindromes(s):
    ___

def palindromic_prefix_ends(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef expand(s):\n    out = []\n    for c in range(2 * len(s) - 1):\n        lo, hi = c // 2, c // 2 + c % 2\n        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:\n            lo, hi = lo - 1, hi + 1\n        out.append(hi - lo - 1)\n    return out\ndef test_manacher():\n    \"against expansion\"\n    assert manacher(\"abba\") == [0, 1, 0, 1, 4, 1, 0, 1, 0] and manacher(\"\") == [0]\n    assert longest_palindrome(\"forgeeksskeegfor\") == \"geeksskeeg\" and longest_palindrome(\"babad\") == \"bab\"\n    assert count_palindromes(\"aaa\") == 6 and palindromic_prefix_ends(\"abacaba\") == [1, 3, 7]\n    rng = random.Random(6)\n    for _ in range(300):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 20)))\n        d = manacher(s)\n        assert d[1:-1] == expand(s), s\n        subs = [(i, j) for i in range(len(s)) for j in range(i + 1, len(s) + 1) if s[i:j] == s[i:j][::-1]]\n        assert count_palindromes(s) == len(subs)\n        best = max(subs, key=lambda p: (p[1] - p[0], -p[0]), default=(0, 0))\n        assert longest_palindrome(s) == s[best[0]:best[1]]\n        assert palindromic_prefix_ends(s) == [L for L in range(1, len(s) + 1) if s[:L] == s[:L][::-1]]\ndef test_speed():\n    \"linear\"\n    s = \"ab\" * 150_000\n    t0 = time.perf_counter()\n    assert len(longest_palindrome(s)) == 299_999 and time.perf_counter() - t0 < 4")

## Warm-up

Every palindrome has a **centre**, and growing outwards from the right centre finds it. Centred on the single character at 7 ("s"), only "s" itself works; centred between positions 7 and 8 (the two "s"s), it grows to "geeksskeeg".

## Expanding around centres

A string of length n has 2n − 1 possible centres: n on characters (odd-length palindromes like "racecar") and n − 1 between characters (even-length ones like "abba"). From each centre, grow outwards while the two ends match. The longest palindrome found anywhere is the longest palindromic substring.

Each expansion is O(n) at worst, so it's **O(n²)** in total, with O(1) extra space. That's fast enough for strings of a few thousand characters, and short to write:

Counting works because every palindromic substring has exactly one centre, and growing from that centre meets it exactly once.

## Palindromes by hashing

With yesterday's prefix hashes you can ask "is `s[i:j]` a palindrome?" in O(1): hash s forwards and the **reversed** string `r = s[::-1]` too. The substring `s[i:j]`, read backwards, is `r[n − j : n − i]`. It's a palindrome exactly when the two hashes agree.

That makes many palindrome questions short. For example, the longest palindrome around each centre can be found by **binary search** on its radius (if a radius works, every smaller one does), giving O(n log n) for the longest palindromic substring.

## Making palindromes

What's the **shortest palindrome** you can make by adding characters to the **front** of s? The answer keeps the longest palindromic **prefix** of s as the middle and puts the reverse of the rest in front: for "abacd", the longest palindromic prefix is "aba", so add "dc" to get "dcabacd".

Finding the longest palindromic prefix is a job for Day 2's prefix function. A prefix p of s is a palindrome exactly when p equals its own reverse, that is, when p is a prefix of s **and** a suffix of reverse(s). So compute the prefix function of `s + "#" + reverse(s)` (the `#` stops a border running across the join): its last value is the length of the longest such p. O(n) in all.

## Rearranging and pairing

Two more questions, both about whole strings:

- **Can these letters be rearranged into a palindrome?** A palindrome pairs up its characters around the middle, so at most **one** character may appear an odd number of times. Count with a `Counter`: O(n).
- **Palindrome pairs**: in a list of distinct words, which ordered pairs (i, j) make `words[i] + words[j]` a palindrome? Checking every pair is O(n² k) for words of length k. Faster: put the words in a dict from word to index. Split each word w into a prefix and a suffix at every position; if the prefix is a palindrome and the reverse of the suffix is another word, then that word followed by w is a palindrome (and symmetrically the other way round). That's O(n k²).

## Manacher's algorithm

Expanding around every centre repeats work: inside a long palindrome, the left half is a mirror image of the right, so the palindromes centred on the right half mirror those on the left, which have already been measured. In 1975 Glenn Manacher turned that into an O(n) algorithm.

The usual modern form handles odd and even lengths together by putting a separator between characters: "abba" becomes "#a#b#b#a#", where every palindrome has odd length. Then, for each position i of the new string t, compute `d[i]`, the radius of the longest palindrome centred there, while remembering the palindrome `[l, r]` that reaches furthest right so far:

- If i is inside `[l, r]`, its mirror is `j = l + r − i`, and `d[i]` is at least `min(d[j], r − i)` (the part of the mirror's palindrome that's guaranteed to be inside the big one).
- Then try to grow beyond that, comparing characters as usual.
- If the palindrome at i now reaches past r, it becomes the new `[l, r]`.

Each successful comparison pushes r to the right, and r never moves left, so there are at most O(n) of them: **O(n)** in total. It's the same trick as yesterday's Z-function (and the reasoning is just as in KMP). A palindrome of radius d in t (counting the centre) corresponds to one of length d − 1 in s. Today's challenge asks you to write it.

## Practice

## Quiz

## Daily challenge

## Before you go

Find the longest palindrome hidden in a sentence you like, by hand, using centres. Tomorrow: the autocomplete project, which puts tries, heaps and this week's string tools together into one program.